In [5]:
import pandas as pd
import time
from tqdm import tqdm
import os

with open("./sras.txt") as f:
    srr_list = f.read().splitlines()

In [6]:
from pysradb.sraweb import SRAweb

db = SRAweb()
results = []
total_srr = len(srr_list)
chunk_size = 128

for i in tqdm(range(0, total_srr, chunk_size), desc="Metadata Fetching", unit="chunk"):
    chunk = srr_list[i:i + chunk_size]
    try:
        df_chunk = db.sra_metadata(chunk, detailed=True)
        if df_chunk is not None and not df_chunk.empty:
            results.append(df_chunk)
        time.sleep(1.5)
        
    except Exception as e:
        tqdm.write(f"오류 발생 (SRR: {chunk[0]} ~): {e}")

Metadata Fetching: 100%|██████████| 29/29 [05:00<00:00, 10.35s/chunk]


In [ ]:
import pandas as pd

processed_results = []
for df in results:
    df_temp = df.copy()
    cols = pd.Series(df_temp.columns)
    for dup_col in cols[cols.duplicated()].unique():
        mask = cols == dup_col
        cols[mask] = [f"{dup_col}_{i}" if i != 0 else dup_col for i in range(sum(mask))]
    df_temp.columns = cols
    processed_results.append(df_temp)
final_df = pd.concat(processed_results, ignore_index=True)
print(f"최종 병합된 데이터의 크기 (행, 열): {final_df.shape}")


최종 병합된 데이터의 크기 (행, 열): (3701, 130)


In [ ]:
final_df.drop_duplicates(subset='run_accession').to_csv("sra_meta_raw.csv", index=False)

,run_accession,study_accession,study_title,experiment_accession,experiment_title,experiment_desc,organism_taxid,organism_name,library_name,library_strategy,...,disease,disease_stage,sample_type,gender,gestational_age_in_weeks_at_delivery,sequencing_batch,gestational_age_in_weeks_at_preeclampsia_onset,treatment,sample_id,disease state
0,ERR15158149,ERP173306,Systematic assessment of RNA-Seq experimental ...,ERX14563493,Illumina MiSeq paired end sequencing,Illumina MiSeq paired end sequencing,9606,Homo sapiens,SAMP165,RNA-Seq,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ERR15158500,ERP173306,Systematic assessment of RNA-Seq experimental ...,ERX14563873,NextSeq 2000 paired end sequencing,NextSeq 2000 paired end sequencing,9606,Homo sapiens,SAMP2456,RNA-Seq,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ERR15158501,ERP173306,Systematic assessment of RNA-Seq experimental ...,ERX14563874,NextSeq 2000 paired end sequencing,NextSeq 2000 paired end sequencing,9606,Homo sapiens,SAMP2646,RNA-Seq,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ERR15158502,ERP173306,Systematic assessment of RNA-Seq experimental ...,ERX14563875,NextSeq 2000 paired end sequencing,NextSeq 2000 paired end sequencing,9606,Homo sapiens,SAMP2681,RNA-Seq,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ERR15158503,ERP173306,Systematic assessment of RNA-Seq experimental ...,ERX14563876,NextSeq 2000 paired end sequencing,NextSeq 2000 paired end sequencing,9606,Homo sapiens,SAMP2705,RNA-Seq,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3696,SRR22561274,SRP411015,Circulating mRNA variants as a potential bioma...,SRX18524631,RNA-Seq of Homo sapiens: Hepatoma plasma HCC011P,RNA-Seq of Homo sapiens: Hepatoma plasma HCC011P,9606,Homo sapiens,Sayeed-1-HCC011-4-3-19_S5,RNA-Seq,...,Hepatoma,NaN,RNA,NaN,NaN,NaN,NaN,<NA>,HCC011P,<NA>
3697,SRR22561275,SRP411015,Circulating mRNA variants as a potential bioma...,SRX18524630,RNA-Seq of Homo sapiens: Hepatoma plasma HCC025P,RNA-Seq of Homo sapiens: Hepatoma plasma HCC025P,9606,Homo sapiens,Sayeed-3-HCC025-4-3-19_S1,RNA-Seq,...,Hepatoma,NaN,RNA,NaN,NaN,NaN,NaN,<NA>,HCC025P,<NA>
3698,SRR22561276,SRP411015,Circulating mRNA variants as a potential bioma...,SRX18524629,RNA-Seq of Homo sapiens: Hepatoma plasma HCC029P,RNA-Seq of Homo sapiens: Hepatoma plasma HCC029P,9606,Homo sapiens,Sayeed-05-HCC029-4-11-19_S4,RNA-Seq,...,Hepatoma,NaN,RNA,NaN,NaN,NaN,NaN,<NA>,HCC029P,<NA>
3699,SRR22561277,SRP411015,Circulating mRNA variants as a potential bioma...,SRX18524628,RNA-Seq of Homo sapiens: Hepatoma plasma HCC031P,RNA-Seq of Homo sapiens: Hepatoma plasma HCC031P,9606,Homo sapiens,Sayeed-11-HCC031-4-11-19_S8,RNA-Seq,...,Hepatoma,NaN,RNA,NaN,NaN,NaN,NaN,<NA>,HCC031P,<NA>
